# Phase 4

### Création du dataset de fraude

In [20]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,recall_score,f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
X, y = make_classification(n_samples=10000,n_features=10,n_informative=5,n_redundant=2,n_clusters_per_class=2,weights=[0.99, 0.01],flip_y=0,class_sep=1.5,
    random_state=42)

X = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(1, 11)])
y = pd.Series(y, name="fraude")

print("Nombre de lignes :", X.shape[0])
print("Nombre de colonnes :", X.shape[1])

print("\nRépartition de la cible :")
print(y.value_counts(normalize=True).sort_index().round(3))

Nombre de lignes : 10000
Nombre de colonnes : 10

Répartition de la cible :
fraude
0    0.99
1    0.01
Name: proportion, dtype: float64


### Séparation train / test

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Train :", len(X_train))
print("Test :", len(X_test))

print("\nRépartition dans le test :")
print(y_test.value_counts(normalize=True).sort_index().round(3))

Train : 8000
Test : 2000

Répartition dans le test :
fraude
0    0.99
1    0.01
Name: proportion, dtype: float64


### Fonction  rapport_metier

In [14]:
def rapport_metier(y_true, y_pred, cout_fn=10, cout_fp=1, nom_modele="Modèle"):
   

    matrice = confusion_matrix(y_true, y_pred, labels=[0, 1])
    VN, FP, FN, VP = matrice.ravel()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    cout_total = FN * cout_fn + FP * cout_fp

    print("=" * 60)
    print(nom_modele)
    print("=" * 60)

    print("Matrice de confusion :")
    print(matrice)

    print("\nDétail :")
    print("VN :", VN)
    print("FP :", FP)
    print("FN :", FN)
    print("VP :", VP)

    print("\nMétriques :")
    print(f"Accuracy  : {accuracy:.3f}")
    print(f"Precision : {precision:.3f}")
    print(f"Recall    : {recall:.3f}")
    print(f"F1-score  : {f1:.3f}")

    print("\nCoût métier :")
    print(f"Coût total = {cout_total}")

    return {
        "modele": nom_modele,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "VN": VN,
        "FP": FP,
        "FN": FN,
        "VP": VP,
        "cout_metier": cout_total
    }
rapport_a = rapport_metier(
    y_true=y_test,
    y_pred=pred_a,
    cout_fn=10,
    cout_fp=1,
    nom_modele="Modèle A - Random Forest"
)



Modèle A - Random Forest
Matrice de confusion :
[[1980    0]
 [  10   10]]

Détail :
VN : 1980
FP : 0
FN : 10
VP : 10

Métriques :
Accuracy  : 0.995
Precision : 1.000
Recall    : 0.500
F1-score  : 0.667

Coût métier :
Coût total = 100


### Créer et entraîner les modèles

In [22]:
modele_a = RandomForestClassifier(n_estimators=200,random_state=42)

modele_b = DecisionTreeClassifier(random_state=42)

modele_a.fit(X_train, y_train)
modele_b.fit(X_train, y_train)

pred_a = modele_a.predict(X_test)
pred_b = modele_b.predict(X_test)

rapport_b = rapport_metier(
    y_true=y_test,
    y_pred=pred_b,
    cout_fn=10,
    cout_fp=1,
    nom_modele="Modèle B - Decision Tree"
)


Modèle B - Decision Tree
Matrice de confusion :
[[1975    5]
 [   7   13]]

Détail :
VN : 1975
FP : 5
FN : 7
VP : 13

Métriques :
Accuracy  : 0.994
Precision : 0.722
Recall    : 0.650
F1-score  : 0.684

Coût métier :
Coût total = 75


#### modèle A

In [23]:
rapport_a = rapport_metier(y_true=y_test,y_pred=pred_a,cout_fn=10,cout_fp=1,
    nom_modele="Modèle A - Random Forest")

Modèle A - Random Forest
Matrice de confusion :
[[1980    0]
 [  10   10]]

Détail :
VN : 1980
FP : 0
FN : 10
VP : 10

Métriques :
Accuracy  : 0.995
Precision : 1.000
Recall    : 0.500
F1-score  : 0.667

Coût métier :
Coût total = 100


#### modèle B

In [24]:
rapport_b = rapport_metier(y_true=y_test, y_pred=pred_b,cout_fn=10,cout_fp=1,
    nom_modele="Modèle B - Decision Tree")

Modèle B - Decision Tree
Matrice de confusion :
[[1975    5]
 [   7   13]]

Détail :
VN : 1975
FP : 5
FN : 7
VP : 13

Métriques :
Accuracy  : 0.994
Precision : 0.722
Recall    : 0.650
F1-score  : 0.684

Coût métier :
Coût total = 75


### Comparaison des deux modèles

#### Tableau comparatif

In [25]:
comparaison_modeles = pd.DataFrame([rapport_a, rapport_b])

display(comparaison_modeles[["modele", "accuracy", "precision", "recall", "f1", "FP", "FN", "cout_metier"]].round(3))

,modele,accuracy,precision,recall,f1,FP,FN,cout_metier
0,Modèle A - Random Forest,0.995,1.000,0.50,0.667,0,10,100
1,Modèle B - Decision Tree,0.994,0.722,0.65,0.684,5,7,75


#### Cas limite

In [26]:
pred_toujours_pas_fraude = np.zeros(len(y_test), dtype=int)

rapport_toujours_pas_fraude = rapport_metier(y_true=y_test,y_pred=pred_toujours_pas_fraude,cout_fn=10,cout_fp=1,
    nom_modele="Modèle limite ")

Modèle limite 
Matrice de confusion :
[[1980    0]
 [  20    0]]

Détail :
VN : 1980
FP : 0
FN : 20
VP : 0

Métriques :
Accuracy  : 0.990
Precision : 0.000
Recall    : 0.000
F1-score  : 0.000

Coût métier :
Coût total = 200


#### Cas adversarial du collègue avec 99 % d’accuracy

##### Comparer le modèle 99 % avec le meilleur coût métier

In [27]:
comparaison_finale = pd.DataFrame([rapport_toujours_pas_fraude,rapport_a,rapport_b])

comparaison_finale = comparaison_finale[
    [
        "modele",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "FP",
        "FN",
        "cout_metier"
    ]
]

display(comparaison_finale.round(3))

,modele,accuracy,precision,recall,f1,FP,FN,cout_metier
0,Modèle limite,0.990,0.000,0.00,0.000,0,20,200
1,Modèle A - Random Forest,0.995,1.000,0.50,0.667,0,10,100
2,Modèle B - Decision Tree,0.994,0.722,0.65,0.684,5,7,75


interpretation 

Les résultats montrent que l’accuracy seule ne suffit pas pour choisir le meilleur modèle.

Le modèle qui prédit toujours “model limite ” obtient une bonne accuracy, car les fraudes sont rares. Par contre, son recall est nul : il ne détecte aucune fraude. Il n’est donc pas adapté.

Pour ce problème, il faut surtout regarder le recall, le F1-score et le coût métier. Comme rater une fraude coûte plus cher qu’une fausse alerte, le meilleur modèle est celui qui réduit le plus le coût total.

Le modèle retenu est donc celui qui a le coût métier le plus faible, même si son accuracy n’est pas forcément la plus élevée.
